# Day 3: Python + LLM APIs for AI Automation
## Insurance Helpdesk Bot Backend — Hands-on Notebook

**Audience:** Beginner to Intermediate UiPath / RPA Developers  
**Duration:** 4 Hours  
**Domain:** Insurance claims, policy Q&A, customer service, underwriting support and UiPath JSON output

## Learning Outcomes

By the end of this notebook, participants will be able to:

- Configure LLM API access safely using environment variables.
- Use the OpenAI Python SDK with an OpenRouter-compatible endpoint.
- Build insurance-domain AI functions for claim summarization, policy Q&A and classification.
- Parse and validate JSON output for UiPath consumption.
- Add retries, error handling and logging.
- Create a FastAPI backend endpoint for an Insurance Helpdesk Bot.

# 0. How to Use This Notebook

1. Read the concept explanation.
2. Run setup cells.
3. Add API key in `.env`.
4. Uncomment API-call lines after key setup.
5. Practice using insurance scenarios.
6. Complete the final backend assignment.

Most API execution lines are commented to avoid failures before configuration.

# 1. Install Required Packages

Required packages:

- `openai`
- `python-dotenv`
- `fastapi`
- `uvicorn`
- `pydantic`
- `requests`

In [ ]:
# Uncomment if required:
# !pip install openai python-dotenv fastapi uvicorn pydantic requests

# 2. Safe API Key Configuration

Create a `.env` file in the same folder as this notebook.

```text
OPENROUTER_API_KEY=your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=openai/gpt-4.1-mini
```

Why this matters:

- API keys should not be hard-coded.
- Environment variables are safer for training and deployment.
- The same code can move from notebook to FastAPI backend.

In [2]:
import os
import json
import time
import logging
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

print("Base URL:", OPENROUTER_BASE_URL)
print("Model:", OPENROUTER_MODEL)
print("API Key Loaded:", bool(OPENROUTER_API_KEY))

Base URL: https://openrouter.ai/api/v1
Model: openai/gpt-4.1-mini
API Key Loaded: True


# 3. Create OpenAI-Compatible Client

The OpenAI SDK can be used with OpenRouter by supplying `base_url`.

```python
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)
```

In [17]:
if not OPENROUTER_API_KEY:
    print("Warning: OPENROUTER_API_KEY is not set. API calls will fail until configured.")

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY
)

# 4. Reusable LLM Call Function

This function centralizes model calls.

Benefits:

- One place to change model name.
- One place to control temperature.
- Easy error handling.
- Easy reuse from UiPath or FastAPI.

In [21]:
#help(client.chat.completions)

In [24]:
def call_llm(prompt, system_message="You are an insurance automation assistant.", temperature=0.5):
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

# Test after API key setup:
print(call_llm("Explain how AI supports insurance claim processing."))

AI supports insurance claim processing in several key ways, enhancing efficiency, accuracy, and customer experience:

1. **Automated Data Extraction:** AI-powered tools use natural language processing (NLP) and optical character recognition (OCR) to automatically extract relevant information from claim forms, documents, and images, reducing manual data entry errors and speeding up the intake process.

2. **Fraud Detection:** Machine learning algorithms analyze patterns and anomalies in claims data to identify potentially fraudulent claims, helping insurers reduce losses and improve risk management.

3. **Claim Triage and Prioritization:** AI systems can assess the complexity and urgency of claims, prioritizing them for faster handling based on factors such as claim type, severity, and historical data.

4. **Damage Assessment:** Computer vision models analyze photos or videos of damaged property or vehicles to estimate repair costs and validate claim legitimacy, enabling quicker and mor

AI supports insurance claim processing in several key ways, enhancing efficiency, accuracy, and customer experience:

1. **Automated Data Extraction:** AI-powered tools use natural language processing (NLP) and optical character recognition (OCR) to automatically extract relevant information from claim forms, documents, and images, reducing manual data entry errors and speeding up the intake process.

2. **Fraud Detection:** Machine learning algorithms analyze patterns and anomalies in claims data to identify potentially fraudulent claims, helping insurers reduce losses and improve risk management.

3. **Claim Triage and Prioritization:** AI systems can assess the complexity and urgency of claims, prioritizing them for faster handling based on factors such as claim type, severity, and historical data.

4. **Damage Assessment:** Computer vision models analyze photos or videos of damaged property or vehicles to estimate repair costs and validate claim legitimacy, enabling quicker and more accurate assessments.

5. **Chatbots and Virtual Assistants:** AI-driven chatbots handle routine customer inquiries, guide claimants through the filing process, and provide real-time updates, improving customer engagement and reducing call center workload.

6. **Predictive Analytics:** AI forecasts claim outcomes and potential costs, helping insurers allocate resources effectively and make informed decisions on claim settlements.

7. **Workflow Automation:** AI integrates with claims management systems to automate repetitive tasks, such as approvals and document routing, minimizing processing time and operational costs.

Overall, AI streamlines insurance claim processing by reducing manual effort, accelerating decision-making, improving accuracy, and enhancing customer satisfaction.

# 5. Insurance Sample Data

In [25]:
health_claim_note = """
Customer Rajesh Sharma submitted a health insurance claim on 12 June 2026.
He was admitted to City Care Hospital for dengue fever treatment.
Total bill amount is INR 85,000. Policy number is POL1001.
Documents submitted include hospital bill, discharge summary and doctor prescription.
Claim is currently pending document verification.
"""

vehicle_claim_note = """
Customer Meena Joshi submitted a vehicle insurance claim after a road accident.
The accident occurred on 18 June 2026 near Mumbai-Pune highway.
Repair estimate is INR 2,40,000. Policy number is VEH2045.
Police report is not attached. Photos are unclear. Policy started 5 days before the accident.
"""

policy_coverage_text = """
Health Secure Plus covers hospitalization expenses up to INR 5,00,000.
Room rent is covered up to INR 5,000 per day.
Pre-hospitalization expenses are covered for 30 days.
Post-hospitalization expenses are covered for 60 days.
Cosmetic treatment, non-prescribed medicines and experimental procedures are excluded.
"""

customer_email = """
Dear Team,
I submitted my hospital claim last week but have not received an update.
Please tell me whether my claim is approved and if any documents are pending.
Regards,
Rajesh Sharma
"""

underwriting_note = """
Applicant is 52 years old and applying for a health policy of INR 10,00,000.
Medical history includes hypertension and diabetes for the last 8 years.
Applicant is a non-smoker. BMI is 31. Previous hospitalization occurred two years ago.
"""

complaint_text = """
I have called your support team five times and nobody has given me a clear answer.
My father's hospital claim is pending for three weeks. We urgently need reimbursement.
Please escalate this immediately.
"""

print("Insurance sample data loaded.")

Insurance sample data loaded.


# 6. Lab 1: Claim Summarization API Client

## Concept

Claim teams receive unstructured notes from emails, call centers, hospital records and UiPath document extraction.

## Hands-on Goal

Generate a clean claim operations summary.

In [27]:
def summarize_claim(claim_note):
    prompt = f"""
You are an insurance claims assistant.
Task: Summarize the raw claim note for claim operations.
Output format:
- Customer Name:
- Policy Number:
- Claim Type:
- Claim Amount:
- Documents Submitted:
- Current Status:
- Recommended Next Action:
Claim Note:
{claim_note}
"""
    return call_llm(prompt)

#print(summarize_claim(health_claim_note))
print(summarize_claim(vehicle_claim_note))

- Customer Name: Meena Joshi  
- Policy Number: VEH2045  
- Claim Type: Vehicle Insurance (Accident)  
- Claim Amount: INR 2,40,000  
- Documents Submitted: Photos (unclear), No police report  
- Current Status: Claim submitted with unclear photos and missing police report; policy started 5 days before the accident  
- Recommended Next Action: Request clear photos and police report; verify policy waiting period and coverage eligibility due to recent policy start date


## Assignment 1: Audience-Based Claim Summary

Create summaries for manager, processor and customer. Notice how tone and detail change.

In [29]:
def summarize_claim_v2(claim_note, audience="processor"):
    if audience == "manager":
        instruction = "Create a short executive summary for a claims manager."
    elif audience == "customer":
        instruction = "Create a simple customer-friendly summary without internal jargon."
    else:
        instruction = "Create a detailed operational summary for a claim processor."
    prompt = f"""
You are an insurance claims assistant.
Task: {instruction}
Claim Note:
{claim_note}
"""
    return call_llm(prompt)

#print(summarize_claim_v2(health_claim_note, "manager"))
print(summarize_claim_v2(health_claim_note, "customer"))

Summary for Rajesh Sharma's Health Insurance Claim:

Rajesh Sharma submitted a health insurance claim on 12 June 2026 for his treatment of dengue fever at City Care Hospital. The total hospital bill is INR 85,000. He provided the hospital bill, discharge summary, and doctor’s prescription as part of his claim. Currently, the claim is under review for document verification.


# 7. Lab 2: Policy Q&A Function

## Concept

A policy Q&A function answers only from provided policy text. It must not invent coverage.

In [11]:
def answer_policy_question(policy_text, question):
    prompt = f"""
You are an insurance policy support assistant.

Rules:
- Answer only using the provided policy text.
- Do not invent coverage details.
- If unavailable, say: "Information not available in provided policy text."
- Keep the answer simple and customer-friendly.

Policy Text:
{policy_text}

Customer Question:
{question}
"""
    return call_llm(prompt)

print(answer_policy_question(policy_coverage_text, "Is room rent covered?"))

Yes, room rent is covered up to INR 5,000 per day.


## Practice 2: Ask Multiple Policy Questions

Try common insurance questions and observe how guardrails work.

In [12]:
policy_questions = [
    "What is the hospitalization coverage limit?",
    "Is cosmetic treatment covered?",
    "Are pre-hospitalization expenses covered?",
    "Is dental treatment covered?"
]

for question in policy_questions:
     print("Question:", question)
     print(answer_policy_question(policy_coverage_text, question))
     print("-" * 60)

Question: What is the hospitalization coverage limit?
The hospitalization coverage limit is up to INR 5,00,000.
------------------------------------------------------------
Question: Is cosmetic treatment covered?
Cosmetic treatment is not covered under your policy.
------------------------------------------------------------
Question: Are pre-hospitalization expenses covered?
Yes, pre-hospitalization expenses are covered for 30 days under your Health Secure Plus policy.
------------------------------------------------------------
Question: Is dental treatment covered?
Information not available in provided policy text.
------------------------------------------------------------


## Assignment 2: Structured Policy Q&A

Return answer, confidence, source sentence and escalation requirement.

In [13]:
def answer_policy_question_structured(policy_text, question):
    prompt = f"""
You are an insurance policy support assistant.
Answer only using the provided policy text.

Return:
Answer:
Confidence: High / Medium / Low
Source Sentence:
Escalation Required: Yes / No

Policy Text:
{policy_text}
Question:
{question}
"""
    return call_llm(prompt)

print(answer_policy_question_structured(policy_coverage_text, "Is dental treatment covered?"))

Answer: Dental treatment is not mentioned as covered in the policy text, so it is not covered.
Confidence: High
Source Sentence: Cosmetic treatment, non-prescribed medicines and experimental procedures are excluded.
Escalation Required: No


# 8. Lab 3: Claim Classification Labels

## Concept

Classification helps UiPath route claim transactions to the right queue.

In [14]:
def classify_claim(claim_note):
    prompt = f"""
You are an insurance claim classification assistant.

Classify using these labels:
Claim Type: Health / Vehicle / Life / Property / Travel / Unknown
Risk Level: Low / Medium / High
Processing Status: Ready for Processing / Missing Documents / Escalation Required

Return only:
Claim Type:
Risk Level:
Processing Status:
Reason:

Claim Note:
{claim_note}
"""
    return call_llm(prompt)

print(classify_claim(vehicle_claim_note))

Claim Type: Vehicle  
Risk Level: High  
Processing Status: Missing Documents  
Reason: Police report is not attached, photos are unclear, and the policy started only 5 days before the accident, indicating potential risk and incomplete documentation.


## Practice 3: Travel Claim Classification

In [15]:
travel_claim_note = """
Customer submitted a travel insurance claim for delayed baggage.
Flight was delayed by 9 hours and baggage was delivered after 2 days.
Customer attached boarding pass and airline delay certificate.
Claim amount is INR 18,000.
"""

print(classify_claim(health_claim_note))
print(classify_claim(vehicle_claim_note))
print(classify_claim(travel_claim_note))

Claim Type: Health  
Risk Level: Medium  
Processing Status: Missing Documents  
Reason: Claim is pending document verification despite submission of key documents; verification must be completed before processing.
Claim Type: Vehicle  
Risk Level: High  
Processing Status: Missing Documents  
Reason: Police report is not attached, photos are unclear, and the policy started only 5 days before the accident, indicating potential risk and incomplete documentation.
Claim Type: Travel  
Risk Level: Low  
Processing Status: Ready for Processing  
Reason: The claim is related to travel insurance for delayed baggage. The customer provided necessary documents (boarding pass and airline delay certificate), and the claim amount is moderate. No missing information or issues requiring escalation.


## Assignment 3: UiPath Queue Routing

Map classification output to queue names.

In [16]:
def route_claim_from_labels(classification_text):
    text = classification_text.lower()
    if "high" in text:
        return "Risk Review Queue"
    if "missing documents" in text:
        return "Document Pending Queue"
    if "health" in text:
        return "Health Queue"
    if "vehicle" in text:
        return "Vehicle Queue"
    if "travel" in text:
        return "Travel Queue"
    return "General Review Queue"

sample_classification = """
Claim Type: Vehicle
Risk Level: High
Processing Status: Missing Documents
Reason: Police report missing and repair estimate is unusually high.
"""
print("Target Queue:", route_claim_from_labels(sample_classification))

Target Queue: Risk Review Queue


# 9. Lab 4: Extract and Parse JSON for UiPath

## Concept

JSON is easier for UiPath to consume than free text.

In [ ]:
def extract_claim_json(claim_note):
    prompt = f"""
You are an insurance automation assistant.
Extract claim information and return valid JSON only.
Do not include explanation outside JSON.

Schema:
{{
  "customer_name": "",
  "policy_number": "",
  "claim_type": "",
  "claim_amount": 0,
  "documents_submitted": [],
  "missing_documents": [],
  "risk_level": "",
  "recommended_action": ""
}}

Claim Note:
{claim_note}
"""
    return call_llm(prompt)

# json_text = extract_claim_json(health_claim_note)
# print(json_text)

## Parse JSON Safely

In [ ]:
def parse_json_safely(json_text):
    try:
        return json.loads(json_text), None
    except json.JSONDecodeError as error:
        return None, str(error)

sample_json_text = '''
{
  "customer_name": "Rajesh Sharma",
  "policy_number": "POL1001",
  "claim_type": "Health",
  "claim_amount": 85000,
  "documents_submitted": ["Hospital Bill", "Discharge Summary", "Doctor Prescription"],
  "missing_documents": [],
  "risk_level": "Low",
  "recommended_action": "Proceed with document verification"
}
'''
parsed_output, error = parse_json_safely(sample_json_text)
print("Error:", error)
print(parsed_output)

# 10. Lab 5: Validate LLM JSON Output

## Concept

Validate before passing output to UiPath.

In [ ]:
required_fields = ["customer_name", "policy_number", "claim_type", "claim_amount", "documents_submitted", "missing_documents", "risk_level", "recommended_action"]
allowed_risk_levels = ["Low", "Medium", "High"]

def validate_claim_json(data):
    errors = []
    for field in required_fields:
        if field not in data:
            errors.append(f"Missing field: {field}")
    if "claim_amount" in data and not isinstance(data["claim_amount"], (int, float)):
        errors.append("claim_amount must be numeric")
    if "documents_submitted" in data and not isinstance(data["documents_submitted"], list):
        errors.append("documents_submitted must be a list")
    if "missing_documents" in data and not isinstance(data["missing_documents"], list):
        errors.append("missing_documents must be a list")
    if "risk_level" in data and data["risk_level"] not in allowed_risk_levels:
        errors.append("risk_level must be Low, Medium or High")
    if "recommended_action" in data and not data["recommended_action"]:
        errors.append("recommended_action must not be empty")
    return errors

print(validate_claim_json(parsed_output))

## Practice 4: Invalid Data Validation

In [ ]:
invalid_claim_data = {
    "customer_name": "Meena Joshi",
    "policy_number": "VEH2045",
    "claim_type": "Vehicle",
    "claim_amount": "Two lakh forty thousand",
    "documents_submitted": "Photos attached",
    "missing_documents": ["Police Report"],
    "risk_level": "Very High",
    "recommended_action": ""
}
for error in validate_claim_json(invalid_claim_data):
    print("-", error)

# 11. Lab 6: Error Handling, Retries and Logging

## Concept

Production API calls may fail due to network issues, rate limits, invalid keys or provider downtime.

In [ ]:
logging.basicConfig(filename="insurance_ai_backend.log", level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def call_llm_with_retry(prompt, max_retries=3, delay_seconds=2):
    for attempt in range(1, max_retries + 1):
        try:
            logging.info(f"LLM call attempt {attempt}; prompt length={len(prompt)}")
            result = call_llm(prompt)
            logging.info("LLM call successful")
            return result
        except Exception as error:
            logging.error(f"LLM call failed on attempt {attempt}: {error}")
            print(f"Attempt {attempt} failed:", error)
            if attempt < max_retries:
                time.sleep(delay_seconds * attempt)
            else:
                return "AI service temporarily unavailable. Route to manual review."

# print(call_llm_with_retry("Summarize a health insurance claim in one sentence."))

# 12. Extra Example 1: Customer Email Response Generator

Generate professional claim-status communication without promising approval.

In [ ]:
def generate_customer_email(customer_message, claim_context):
    prompt = f"""
You are a professional insurance customer support assistant.
Draft a polite email reply.
Rules:
- Do not promise claim approval.
- Mention current known status.
- Ask for missing documents if needed.
- Keep tone professional and helpful.
Customer Message:
{customer_message}
Claim Context:
{claim_context}
"""
    return call_llm(prompt)

# print(generate_customer_email(customer_email, health_claim_note))

## Assignment 5: Customer SMS Generator

Create a short SMS update under 30 words.

In [ ]:
def generate_customer_sms(customer_message, claim_context):
    prompt = f"""
Create a short insurance claim status SMS.
Rules:
- Maximum 30 words
- Do not promise approval
- Mention next action clearly
Customer Message:
{customer_message}
Claim Context:
{claim_context}
"""
    return call_llm(prompt)

# print(generate_customer_sms(customer_email, health_claim_note))

# 13. Extra Example 2: Underwriting Notes Assistant

AI supports underwriting review but should not make final approval decisions.

In [ ]:
def generate_underwriting_notes(note):
    prompt = f"""
You are an insurance underwriting support assistant.
Create structured underwriting notes.
Rules:
- Do not approve or reject the applicant.
- Identify risk factors neutrally.
- Suggest additional information needed.
Return:
- Applicant Summary
- Risk Factors
- Missing Information
- Recommended Underwriter Review Points
Applicant Note:
{note}
"""
    return call_llm(prompt)

# print(generate_underwriting_notes(underwriting_note))

## Assignment 6: Underwriting JSON Output

In [ ]:
def generate_underwriting_json(note):
    prompt = f"""
You are an insurance underwriting support assistant.
Return valid JSON only.
Schema:
{{
  "applicant_age": 0,
  "medical_risk_factors": [],
  "lifestyle_factors": [],
  "missing_information": [],
  "recommended_review": ""
}}
Applicant Note:
{note}
"""
    return call_llm(prompt)

# print(generate_underwriting_json(underwriting_note))

# 14. Extra Example 3: Missing Document Checklist Generator

In [ ]:
def generate_missing_document_checklist(claim_note):
    prompt = f"""
You are an insurance claims document verification assistant.
Identify submitted and missing documents.
Return:
- Claim Type
- Documents Submitted
- Likely Missing Documents
- Reason Each Missing Document May Be Needed
- Next Action for Claim Processor
Claim Note:
{claim_note}
"""
    return call_llm(prompt)

# print(generate_missing_document_checklist(vehicle_claim_note))

## Assignment 7: Document Checklist JSON for UiPath

In [ ]:
def generate_document_checklist_json(claim_note):
    prompt = f"""
You are an insurance claims document verification assistant.
Return valid JSON only.
Schema:
{{
  "claim_type": "",
  "submitted_documents": [],
  "missing_documents": [],
  "next_action": "",
  "route_to_queue": ""
}}
Claim Note:
{claim_note}
"""
    return call_llm(prompt)

# print(generate_document_checklist_json(vehicle_claim_note))

# 15. Extra Example 4: Complaint Priority Classifier

In [ ]:
def classify_complaint_priority(complaint):
    prompt = f"""
You are an insurance customer service triage assistant.
Classify the complaint.
Return:
- Sentiment: Positive / Neutral / Negative
- Urgency: Low / Medium / High
- Category: Claim Delay / Policy Issue / Payment Issue / Other
- Escalation Required: Yes / No
- Suggested Response Tone
- Reason
Complaint:
{complaint}
"""
    return call_llm(prompt)

# print(classify_complaint_priority(complaint_text))

## Assignment 8: Complaint JSON for Queue Routing

In [ ]:
def classify_complaint_json(complaint):
    prompt = f"""
You are an insurance support triage assistant.
Return valid JSON only.
Schema:
{{
  "sentiment": "",
  "urgency": "",
  "category": "",
  "escalation_required": "",
  "target_queue": "",
  "suggested_response_tone": ""
}}
Complaint:
{complaint}
"""
    return call_llm(prompt)

# print(classify_complaint_json(complaint_text))

# 16. FastAPI Backend for Insurance Helpdesk Bot

## Architecture

```text
UiPath Robot → HTTP Request → FastAPI Backend → LLM API → JSON Response → UiPath Workflow
```

Minimum endpoint:

```text
POST /claim-assist
```

In [ ]:
fastapi_app_code = """
from fastapi import FastAPI
from pydantic import BaseModel
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)
app = FastAPI(title="Insurance Helpdesk Bot Backend")

class ClaimRequest(BaseModel):
    claim_note: str

def call_llm(prompt):
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": "You are an insurance automation assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

@app.post("/claim-assist")
def claim_assist(request: ClaimRequest):
    prompt = f"""
    Analyze the insurance claim note and return:
    - Claim Summary
    - Claim Type
    - Risk Level
    - Missing Documents
    - Recommended Action

    Claim Note:
    {request.claim_note}
    """
    result = call_llm(prompt)
    return {"status": "success", "ai_response": result}
"""
Path("app.py").write_text(fastapi_app_code, encoding="utf-8")
print("app.py created.")
print("Run: uvicorn app:app --reload")
print("Open: http://127.0.0.1:8000/docs")

## Test FastAPI Endpoint using Requests

Run this only after the server is running.

In [ ]:
import requests

test_payload = {"claim_note": health_claim_note}

# response = requests.post("http://127.0.0.1:8000/claim-assist", json=test_payload)
# print(response.status_code)
# print(response.json())

# 17. Final Assignment: Insurance Helpdesk Bot Backend

Build a backend with:

1. Claim summarization
2. Policy Q&A
3. Claim classification
4. Risk indicator extraction
5. Missing document checklist
6. JSON output for UiPath
7. Error handling
8. Logging
9. FastAPI endpoint

## Optional Endpoints

```text
POST /policy-qa
POST /classify-claim
POST /document-checklist
POST /customer-email
POST /complaint-priority
```

# 18. Final Checklist

| Requirement | Completed |
|---|---|
| `.env` configuration | ☐ |
| OpenAI-compatible client | ☐ |
| Claim summarization | ☐ |
| Policy Q&A | ☐ |
| Claim classification | ☐ |
| JSON parsing | ☐ |
| JSON validation | ☐ |
| Retry and logging | ☐ |
| FastAPI endpoint | ☐ |
| Swagger UI test | ☐ |
| UiPath HTTP Request ready | ☐ |

# 19. Discussion Questions

1. Why should API keys not be hard-coded?
2. Why is JSON better than plain text for UiPath?
3. What can go wrong with LLM outputs?
4. Where is human review mandatory?
5. Which insurance workflows are safe for automation?
6. What should be logged for audit?
7. How can UiPath call this backend?

# 20. Day 3 Summary

You have completed:

- OpenAI SDK and OpenRouter-compatible setup
- Environment-variable configuration
- Claim summarization API client
- Policy Q&A
- Claim classification
- JSON extraction and validation
- Retry and logging
- Customer email generation
- Underwriting notes assistant
- Missing document checklist
- Complaint priority classifier
- FastAPI backend for Insurance Helpdesk Bot

This backend becomes the foundation for LangChain, LangGraph, MCP and UiPath integration in upcoming sessions.